# WAVEGO Pro HTTP Bypass API

Control and read the robot over HTTP — no web UI needed. The main app serves an unauthenticated API on port 5000 (LAN-only trust model).

**Safety first**: movement commands are real. The `T:111` gait command is *level-based* — the robot keeps moving until it receives a stop. Put the robot on a stand for first experiments, and always end scripts with a stop.

```text
POST /api/cmd   {"T":111,"FB":1,"LR":0}   gait vector (FB/LR each -1|0|+1)
GET  /api/status                            sensors + video + CV snapshot
GET  /api/telemetry                         joints @1Hz, battery, host errors
GET/POST /api/cv                            CV mode switch (person follow!)
GET/POST /api/config                        whitelisted runtime settings
GET  /api/vlm                               VLM goal-session status
```

In [ ]:
import requests, json

BASE = "http://127.0.0.1:5000"

def post(path, payload=None, **kw):
    r = requests.post(BASE + path, json=payload, timeout=5, **kw)
    print(r.status_code, r.text[:300])
    return r

def get(path, **kw):
    r = requests.get(BASE + path, timeout=5, **kw)
    print(r.status_code, r.text[:300])
    return r

get("/api/status")

## Gait movement (T:111)

| FB | LR | result |
|----|----|--------|
| 1 | 0 | forward |
| -1 | 0 | backward |
| 0 | -1 | turn left |
| 0 | 1 | turn right |
| 0 | 0 | **stop** |

In [ ]:
import time

post("/api/cmd", {"T": 111, "FB": 0, "LR": 0})   # stop first
time.sleep(0.5)
post("/api/cmd", {"T": 111, "FB": 1, "LR": 0})   # walk forward
time.sleep(1.0)
post("/api/cmd", {"T": 111, "FB": 0, "LR": 0})   # STOP - always end with this

## Synchronous queries

Query commands (T:106 joint positions, T:207 battery) reply with a negative `T`. Use `?sync_ms=` to short-poll for the reply.

In [ ]:
post("/api/cmd?sync_ms=500", {"T": 207})    # -> response: {"T": -207, "voltage": 7.xx}

## CV modes and person following

In [ ]:
get("/api/cv")                                   # current mode
post("/api/cv", {"mode": "person"})              # start FOLLOW (robot may move!)
# post("/api/cv", {"mode": "none"})              # stop following
post("/api/cv", {"mode": "none", "motion_lock": True})   # hard gate: no CV motion

## Runtime settings (whitelisted, persisted to config.yaml)

In [ ]:
get("/api/config")
post("/api/config", {"video.default_quality": 40})

## Telemetry snapshot (joints, battery, host errors)

In [ ]:
d = get("/api/telemetry").json()
labels = ["LF_A", "LF_B", "LF_H", "LH_A", "LH_B", "LH_H",
          "RF_A", "RF_B", "RF_H", "RH_A", "RH_B", "RH_H"]
fb = (d.get("joints") or {}).get("fb")
if fb:
    for name, v in zip(labels, fb):
        print(f"{name:5s} {v:4d} " + "#" * (v * 40 // 1023))
else:
    print("no joint frame yet (poll runs at 1 Hz inside the app)")
print("battery:", d.get("battery"), "V | host errors:", len(d.get("errors", [])))

## VLM goal sessions (voice / dashboard / API triggers)

The VLM service drives real motion until a goal is verified reached. Status only below; see the Telemetry tab and `ugv_rpi/README.md` for the full picture.

In [ ]:
get("/api/vlm")   # session state, queue, online verifier
# post("/api/vlm", {"goal": "find a ball and stop in front of it"})   # starts motion!
# post("/api/vlm/stop", {})                                            # kill switch